# 05 – Vergleich: Basis · nur Stufe 1 · Stufe 1+2
Klärt, ob Stufe 2 (GRPO) etwas bringt und ob Stufe 1 schon wirkt.

- **40 Fragen**, alle mit fairer Referenz: 10 allgemein, 15 Business, 15 Business-Evolution. Business kam im Training nie vor.
- Pro Frage und Variante **4 Samples**, in einem Generierungs-Batch.
- **Blinder Paarvergleich** durch ein Modell einer **anderen Familie** (Llama-3.1-8B, nicht der Trainings-Judge):
  1. Jede Antwort wird auf ihre **Kernidee** (≤ 2 Sätze) reduziert. Das neutralisiert Format und Länge.
  2. Je zwei Kernideen werden in **beiden Reihenfolgen** verglichen. Ein Sieg zählt nur, wenn beide Urteile übereinstimmen.
  3. Drei Kriterien: **originell**, **umsetzbar**, **gesamt** (überraschend UND umsetzbar).
- Ergebnis: Siegquoten mit 95%-Bootstrap-Intervall (Einheit = Frage), Entscheidungsregel und 20 Paare für deine eigene Blindbewertung.

**Kaggle:** Inputs = Stufe-1-Lauf (`adapter_sft`) + GRPO-Lauf (`adapter_grpo`) · GPU T4 x2 · Internet an · Save & Run All (Commit).
**Dauer ~3–3,5 h.** Ein Zeitwächter stoppt die Generierung nach 2,6 h, die drei Varianten sind dann immer gleich vollständig.
Ausgabe in `vergleich/`.

In [ ]:
import time; START = time.time()
# === Setup ===
REPO_URL = "https://github.com/Lommel123/creativ-model.git"
import os, sys, glob, json, random, subprocess
os.chdir("/kaggle/working")
if not os.path.exists("repo"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "repo"], check=True)
%pip install -q unsloth pyyaml sentence-transformers
sys.path.insert(0, "/kaggle/working/repo/scripts")
import yaml, numpy as np, torch
import compare_lib as CL
from judge_client import Judge
CFG = yaml.safe_load(open("repo/config.yaml")); C = CFG["compare"]
OUT = "/kaggle/working/vergleich"; os.makedirs(OUT, exist_ok=True)
def el(): return f"{(time.time() - START) / 3600:.2f} h"
print(subprocess.run(["git", "-C", "repo", "log", "--oneline", "-1"], capture_output=True, text=True).stdout, "GPUs:", torch.cuda.device_count())

In [ ]:
# === Inputs: adapter_sft (Stufe 1) und adapter_grpo (Stufe 1+2) ===
def find(name):
    f = glob.glob(f"/kaggle/input/**/{name}/adapter_model.safetensors", recursive=True)
    assert f, f"{name} nicht gefunden – rechts 'Add Input → Your Work' den passenden Lauf anhängen"
    if len(f) > 1: print(f"Hinweis: mehrere {name} gefunden, nehme {f[0]}")
    return f[0]
SFT_W, GRPO_W = find("adapter_sft"), find("adapter_grpo")
print("Stufe 1  :", SFT_W); print("Stufe 1+2:", GRPO_W)

ld = lambda p: [json.loads(l) for l in open(p, encoding="utf-8")]
QS = CL.select_prompts(ld("repo/eval/eval_prompts.jsonl"), ld("repo/eval/business_holdout.jsonl"),
                       ld("repo/eval/business_evolution.jsonl"), C["prompts"])
print(len(QS), "Fragen:", {k: sum(q["kind"] == k for q in QS) for k in CL.KINDS})

# Judge (andere Modellfamilie) lädt schon jetzt im Hintergrund auf GPU 1, während GPU 0 generiert
JGPU = 1 if torch.cuda.device_count() > 1 else 0
JUDGE = Judge(C["judge_model"], gpu=JGPU, wait=False)

In [ ]:
# === Modell: Basis + ein LoRA-Slot; Gewichte von Stufe 1 / Stufe 1+2 werden je Frage eingetauscht ===
from unsloth import FastLanguageModel
from peft import set_peft_model_state_dict
from safetensors.torch import load_file
L = CFG["lora"]
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CFG["base_model"], max_seq_length=CFG["max_seq_length"], load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=L["r"], lora_alpha=L["alpha"], lora_dropout=L["dropout"],
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"], random_state=42)
W = {"sft": load_file(SFT_W), "grpo": load_file(GRPO_W)}
for k, sd in W.items():
    r = set_peft_model_state_dict(model, sd)
    miss = [m for m in r.missing_keys if "lora_" in m]
    assert not miss and not r.unexpected_keys, f"{k}: Adapter passt nicht zum Basismodell"
    print(f"✓ {k}: {len(sd)} Tensoren passen")
FastLanguageModel.for_inference(model)
for m in model.modules():   # Absicherung (Fehler aus Stufe-2-V1)
    if type(m).__name__.endswith(("ForCausalLM", "Model")) and not hasattr(m, "max_seq_length"):
        m.max_seq_length = CFG["max_seq_length"]

def chat(messages):
    kw = dict(tokenize=False, add_generation_prompt=True)
    try: return tokenizer.apply_chat_template(messages, enable_thinking=False, **kw)
    except TypeError: return tokenizer.apply_chat_template(messages, **kw)

@torch.no_grad()
def sample4(q, seed):
    torch.manual_seed(seed)
    enc = tokenizer(chat([{"role": "user", "content": q}]), return_tensors="pt").to("cuda:0")
    n = C["samples"]
    out = model.generate(input_ids=enc.input_ids.repeat(n, 1), attention_mask=enc.attention_mask.repeat(n, 1),
                         max_new_tokens=C["max_new_tokens"], do_sample=True, temperature=C["temperature"], top_p=0.95)
    return [tokenizer.decode(o[enc.input_ids.shape[1]:], skip_special_tokens=True).strip() for o in out]

def answer(arm, q, seed):
    if arm == "base":
        with model.disable_adapter():
            return sample4(q, seed)
    set_peft_model_state_dict(model, W[arm])
    return sample4(q, seed)

print("Schnelltest:", answer("sft", "Wie hält man Brot länger frisch?", 0)[0][:200].replace("\n", " "), "…|", el())

In [ ]:
# === Generieren: je Frage alle 3 Varianten × 4 Samples; Zeitwächter stoppt zwischen Fragen ===
outputs, t_gen, budget = {}, time.time(), C["gen_budget_hours"] * 3600
for n, q in enumerate(QS):
    done = time.time() - t_gen
    if n >= 3 and done + done / n > budget:
        print(f"⏱ Zeitbudget: stoppe nach {n} von {len(QS)} Fragen (alle Varianten gleich vollständig)"); break
    ans = {arm: answer(arm, q["prompt"], C["seed"] + 100 * q["qid"] + k) for k, arm in enumerate(CL.ARMS)}
    outputs[q["qid"]] = {**q, "n_samples": C["samples"], "answers": ans}
    CL.save_json(outputs, f"{OUT}/antworten.json")          # Zwischenstand sichern
    print(f"{n+1:2d}/{len(QS)} {q['kind']:18s} {(time.time()-t_gen)/(n+1):5.0f} s/Frage | {el()}")
print(f"Fertig: {len(outputs)} Fragen × 3 Varianten × {C['samples']} | {el()}")
del model; torch.cuda.empty_cache()

In [ ]:
# === Judge-Schritt 1: jede Antwort auf ihre Kernidee reduzieren (formatneutral) ===
try:
    JUDGE._ensure_ready(); print("Judge:", C["judge_model"])
except Exception as e:
    print("⚠ Judge nicht ladbar:", e, "→ Rückfall auf", C["judge_fallback"])
    JUDGE.close(); JUDGE = Judge(C["judge_fallback"], gpu=JGPU)
keys = [(qid, arm, i) for qid, o in outputs.items() for arm in CL.ARMS for i in range(o["n_samples"])]
texts = JUDGE.text([CL.SUMMARY_PROMPT.format(q=outputs[q]["prompt"], a=outputs[q]["answers"][a][i][:2500]) for q, a, i in keys],
                   max_new_tokens=100, bs=8)
summaries = {k: t for k, t in zip(keys, texts)}
CL.save_json([{"qid": q, "arm": a, "i": i, "kernidee": summaries[(q, a, i)]} for q, a, i in keys], f"{OUT}/kernideen.json")
print(len(summaries), "Kernideen |", el())
ex = next(iter(outputs))
for arm in CL.ARMS: print(f"  {CL.ARM_NAME[arm]:10s}: {summaries[(ex, arm, 0)][:220]}")

In [ ]:
# === Judge-Schritt 2: blinder Paarvergleich, 3 Kriterien, BEIDE Reihenfolgen ===
comps = CL.build_comparisons(outputs, summaries)
raw = JUDGE.text([c["prompt"] for c in comps], max_new_tokens=3, bs=16)
verdicts = [CL.parse_verdict(t) for t in raw]
CL.save_json([{**{k: c[k] for k in ("qid", "x", "y", "i", "crit", "order")}, "roh": r, "urteil": v}
              for c, r, v in zip(comps, raw, verdicts)], f"{OUT}/urteile.json")
res, consistency, n_valid, n_total = CL.outcomes(comps, verdicts)
print(f"{len(comps)} Urteile · unlesbar: {sum(v is None for v in verdicts)} · Positionskonsistenz: {consistency:.0%} | {el()}")
JUDGE.close()

In [ ]:
# === Auswertung ===
kinds_of = {q: o["kind"] for q, o in outputs.items()}
rows = CL.win_table(res, kinds_of)

# Zusatz: Embedding-Abstand der Kernidee zur naheliegenden Lösung (faire Referenz bei allen Fragen)
orig_rows = []
try:
    from sentence_transformers import SentenceTransformer
    emb = SentenceTransformer(C["embed_model"], device=f"cuda:{JGPU}")
    e = lambda t: emb.encode(f"query: {t}", normalize_embeddings=True)
    dist = {(q, a): np.mean([1 - float(e(summaries[(q, a, i)]) @ e(o["obvious"])) for i in range(o["n_samples"])])
            for q, o in outputs.items() for a in CL.ARMS}
    for x, y in CL.PAIRS:
        for kind in CL.KINDS + ("alle",):
            d = [dist[(q, y)] - dist[(q, x)] for q in outputs if kind == "alle" or kinds_of[q] == kind]
            if len(d) >= 3:
                m, lo, hi = CL.bootstrap(d)
                orig_rows.append({"vergleich": f"{CL.ARM_NAME[y]} vs. {CL.ARM_NAME[x]}", "gruppe": kind, "n": len(d),
                                  "delta": m, "lo": lo, "hi": hi})
except Exception as ex:
    print("⚠ Embedding-Zusatz übersprungen:", ex)

dec = CL.decision(rows)
md = CL.to_markdown(rows, consistency, n_valid, n_total, dec, orig_rows)
open(f"{OUT}/ergebnis.md", "w", encoding="utf-8").write(md)
n_blind = CL.blind_files(outputs, summaries, f"{OUT}/blind_bewertung.csv", f"{OUT}/blind_schluessel.csv")
print(md)
print(f"Für dich: {n_blind} Paare in vergleich/blind_bewertung.csv (Schlüssel separat) | Gesamtlaufzeit {el()}")